# 01 · Exploratory data analysis: Snappfood reviews

Exploration only: every function used here lives in `src/reviewlens/`. Run `reviewlens data download` and `reviewlens data prepare` first.

Questions:
1. Are the classes balanced?
2. How long are the reviews?
3. Which words and emojis separate positive from negative reviews?
4. What does normalization change?
5. How many labeled samples does each label fraction leave us?

In [ ]:
import os
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Run from the project root so the relative paths in the config resolve.
if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

from reviewlens.config import load_config
from reviewlens.data.load import read_raw_csv
from reviewlens.data.prepare import load_prepared
from reviewlens.data.split import labeled_mask, mask_labels
from reviewlens.preprocessing import normalize_text

cfg = load_config()
LABELS = cfg.task.labels
train, test = load_prepared(cfg)
print(f'train={len(train):,} rows  test={len(test):,} rows')
train.head()

## 1. Class balance

In [ ]:
balance = pd.DataFrame({
    split: frame['label'].map(dict(enumerate(LABELS))).value_counts()
    for split, frame in [('train', train), ('test', test)]
})
balance['train_%'] = (100 * balance['train'] / balance['train'].sum()).round(1)
balance['test_%'] = (100 * balance['test'] / balance['test'].sum()).round(1)
balance

## 2. Review length (words)

In [ ]:
train['n_words'] = train['text'].str.split().str.len()
train.groupby(train['label'].map(dict(enumerate(LABELS))))['n_words'].describe().round(1)

In [ ]:
clip = int(train['n_words'].quantile(0.99))
fig, ax = plt.subplots(figsize=(8, 4))
for label_id, name in enumerate(LABELS):
    lengths = train.loc[train['label'] == label_id, 'n_words'].clip(upper=clip)
    ax.hist(lengths, bins=40, alpha=0.6, label=name)
ax.set(xlabel=f'words per review (clipped at 99th percentile = {clip})',
       ylabel='reviews', title='Review length by class')
ax.legend()
plt.show()

## 3. Words that separate the classes

Plain frequency is dominated by words common to both classes, so words are ranked by a smoothed **log-odds ratio**: `log((count_in_class + 1) / (count_in_other + 1))`, among words seen at least 50 times.

Tokens are runs of word characters **or half-spaces** (`[\w\u200c]+`). Splitting on spaces would glue punctuation to words (`افتضاح.` ≠ `افتضاح`), and the plain `\w+` pattern would split half-space words (`بی‌کیفیت` → `بی` + `کیفیت`).

In [ ]:
TOKEN_RE = re.compile(r'[\w\u200c]+')

def word_counts(texts):
    return Counter(token.strip('\u200c') for text in texts for token in TOKEN_RE.findall(text))

counts = {i: word_counts(train.loc[train['label'] == i, 'text']) for i in range(len(LABELS))}
vocab = [w for w in counts[0].keys() | counts[1].keys() if counts[0][w] + counts[1][w] >= 50]
log_odds = pd.Series({w: np.log((counts[1][w] + 1) / (counts[0][w] + 1)) for w in vocab})

top = 15
pd.DataFrame({
    'most negative': log_odds.nsmallest(top).index,
    'neg log-odds': log_odds.nsmallest(top).round(2).values,
    'most positive': log_odds.nlargest(top).index,
    'pos log-odds': log_odds.nlargest(top).round(2).values,
})

In [ ]:
print(f'vocabulary size (all tokens): {len(counts[0].keys() | counts[1].keys()):,}')
print(f'tokens seen >= 50 times:      {len(vocab):,}')

### Emojis

In [ ]:
EMOJI_RE = re.compile('[\U0001F300-\U0001FAFF\u2600-\u27BF]')
has_emoji = train['text'].str.contains(EMOJI_RE)
print(f'reviews with at least one emoji: {has_emoji.sum()} ({has_emoji.mean():.1%})')

## 4. What normalization changes

In [ ]:
raw = read_raw_csv(cfg.paths.raw_dir / 'train.csv', cfg.data, cfg.task)
raw['normalized'] = raw['text'].map(normalize_text)
changed = raw[raw['text'] != raw['normalized']]
print(f'normalization changed {len(changed) / len(raw):.1%} of raw reviews')
with pd.option_context('display.max_colwidth', 120):
    display(changed[['text', 'normalized']].sample(6, random_state=0))

## 5. Labeled samples per label fraction

`mask_labels` hides all but a stratified fraction of the training labels (seed 0 shown). The remaining rows stay in the training set as unlabeled data for the semi-supervised methods.

In [ ]:
rows = []
for fraction in cfg.experiment.label_fractions:
    masked = mask_labels(train['label'].to_numpy(), fraction, seed=0)
    visible = labeled_mask(masked)
    rows.append({
        'fraction': fraction,
        'labeled': int(visible.sum()),
        'unlabeled': int((~visible).sum()),
        **{f'labeled_{name}': int((masked == i).sum()) for i, name in enumerate(LABELS)},
    })
pd.DataFrame(rows)

## Findings and consequences for the next phases

1. **Balanced classes** (≈50/50 in train and test). Accuracy and macro-F1 will be close;
   macro-F1 stays the primary metric because it remains meaningful if a topic task is unbalanced.
2. **Negative reviews are longer** (median 16 vs 12 words). Complaints explain what went wrong,
   praise is short ("خیلی خوشمزه بود"). Length alone carries some signal.
3. **Clear sentiment vocabulary**: negative reviews use words like افتضاح, بدترین, فاجعه, پشیمونم;
   positive ones چسبید, خوشمزه‌ترین, بی‌نظیر, ممنونم. A linear model on TF-IDF should
   already work well, and its coefficients will give readable "top contributing words".
4. **Spelling variants split features**: بدترین / بد‌ترین, آخرین / اخرین, بی‌نظیر / بینظیر
   (informal writing, inconsistent half-spaces). Options for phase 3: character n-gram TF-IDF,
   which matches the shared substrings, compared against word n-grams.
5. **Tokenization must keep half-spaces**: scikit-learn's default token pattern (`\w\w+`)
   treats the half-space as a separator and would split بی‌کیفیت into بی + کیفیت.
   Phase 3 needs a token pattern such as `[\w‌]+`.
6. **No emojis** in this dataset (they were removed upstream), so keeping emojis in the
   normalizer has no effect here, but stays correct for other review sources.
7. **Very few labels at 1 %**: about 600 labeled reviews (300 per class) against ~59,800 unlabeled.
   That is the setting where semi-supervised learning has the most room to help.
